# CSV ingest — PVSyst only

Finds the PVSyst CSVs new in `Files/Raw PVSyst1/{plant}/`.
It parses them with a 10-line header (`PVSYST x.y.z; Project; Geographical Site; ...; date`),
and uploads them to `raw_pvsyst` and `proc_pvsyst` (both tables are shared across plants;
the column `plant_code` discriminates and the tables are partitioned by it).

**SCADA is not consumed by a csv** — `scada_ingest.ipynb`
reads this directly from a processed table in OneLake.

**Idempotent**

In [4]:
plant_codes   = None     
files          = None     # lista de ficheros concretos (coma o JSON). Vacio = autodescubrir
force_reingest = False    # True to rebuild from scratch (only dev)
time_shift_min = 0    # OVERRIDE manual (min, multiplo de 15). El time-shift
                      # principal viene de la tabla T08_time_shifts por planta
                      # y periodo; este escalar solo se usa si la planta NO
                      # tiene reglas en T08 (compat. legacy).


In [5]:
import re
import json
import notebookutils
from io import StringIO
from pathlib import Path
from datetime import datetime
import pandas as pd
from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable

# ---------------------------------------------------------
# 1. (Paths for Python: openpyxl, pandas, pathlib)
# ---------------------------------------------------------
LOCAL_FILES = '/lakehouse/default/Files'
CONFIG_XLSX = f'{LOCAL_FILES}/Raw PVSyst1/CONFIG/plants_config.xlsx'

# ---------------------------------------------------------
# 2. (Paths for notebookutils and Spark)
# ---------------------------------------------------------
ONELAKE_FILES      = 'Files'
ONELAKE_RAW_ROOT   = f'{ONELAKE_FILES}/Raw PVSyst1'
ONELAKE_OUTPUT_DIR = f'{ONELAKE_FILES}/output'

ONELAKE_EXCEL_PATH = "Files/Raw PVSyst1/CONFIG/plants_config.xlsx"
LOCAL_TEMP_PATH = "file:///tmp/plants_config_temp.xlsx"
REAL_LOCAL_PATH = "/tmp/plants_config_temp.xlsx"

## 1 · Discover pending CSVs

In [6]:
def _norm_plant_codes(v):
    """Normaliza plant_codes a lista (una o varias). None/'' = todas.
    Acepta lista/tupla, JSON array o cadena separada por comas."""
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    try:
        import json as _json
        parsed = _json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, TypeError):
        pass
    return [p.strip() for p in s.split(',') if p.strip()]


def discover_csvs(root: str, plant_codes_list):
    """Escanea subcarpetas de root y devuelve los CSV. Salta carpetas que NO son
    plantas (inbox, CONFIG, LOGS, etc.) para evitar procesar ficheros que cuelgan
    de la biblioteca SharePoint pero no son outputs PVsyst."""
    # Carpetas reservadas dentro de Documentos/ (shortcut Raw PVSyst1) que NO son plantas.
    SKIP_FOLDERS = {'INBOX', 'CONFIG', 'LOGS', 'PERFORMANCE', 'TRAZABILIDAD', '_LOGS'}

    candidates = []
    for pf in notebookutils.fs.ls(root):
        if not pf.isDir:
            continue
        folder_plant = pf.name.upper()
        if folder_plant in SKIP_FOLDERS:
            continue
        if plant_codes_list and folder_plant not in plant_codes_list:
            continue
        for f in notebookutils.fs.ls(pf.path):
            if not f.name.lower().endswith('.csv'):
                continue
            candidates.append({
                'plant_code': folder_plant,
                'file_kind':  'pvsyst',
                'full_path':  f.path,
                'filename':   f.name,
                'file_size':  f.size,
            })
    return candidates

files_list = _norm_plant_codes(files)   # reutilizamos el parser de listas (coma / JSON / lista)

if files_list:
    # MODO EXPLICITO: ingerir exactamente los ficheros indicados (por nombre o ruta),
    # aunque ya estuvieran ingeridos.
    _wanted = {Path(f).name.strip().upper() for f in files_list}
    _all = discover_csvs(ONELAKE_RAW_ROOT, [])   # sin filtro de planta
    candidates = [c for c in _all if c['filename'].upper() in _wanted]
    _missing = _wanted - {c['filename'].upper() for c in candidates}
    if _missing:
        print(f"  [WARN] Ficheros no encontrados en Raw PVSyst1: {sorted(_missing)}")
    print(f'Modo ficheros explicitos: {len(candidates)}/{len(files_list)} encontrados')
else:
    # MODO NORMAL (sin cambios): autodescubrir plantas activas; incremental salvo force_reingest.
    candidates = discover_csvs(ONELAKE_RAW_ROOT, [p.upper() for p in _norm_plant_codes(plant_codes)])
    print(f'Encontrados {len(candidates)} CSV de PVSyst en SharePoint')

In [7]:
if force_reingest or files_list:
    to_ingest = candidates
    _why = 'force_reingest=True' if force_reingest else 'ficheros explicitos'
    print(f'{_why} → procesando los {len(to_ingest)} candidatos (sin filtro de ya-ingeridos)')
else:
    already_df = spark.sql("SELECT file_path FROM ingested_files WHERE status = 'ok'")
    already = {r['file_path'] for r in already_df.collect()}
    to_ingest = [c for c in candidates if c['full_path'] not in already]
    print(f'{len(already)} ya ingeridos · {len(to_ingest)} nuevos por procesar')

for c in to_ingest:
    print(f"  {c['plant_code']} · {c['filename']}")


## 2 · Verify plants against `T01_plants` in config excel

In [8]:
import openpyxl


def read_excel_table(filepath: str, table_name: str) -> pd.DataFrame:
    """Busca una tabla por su nombre en el Excel y devuelve sus datos."""
    # En Fabric, /lakehouse/default/Files/ no es accesible por Python I/O.
    # Copiamos primero a /tmp/ via notebookutils antes de abrir con openpyxl.
    local_path = filepath
    if filepath.startswith('/lakehouse/'):
        local_path = f'/tmp/{Path(filepath).name}'
        rel_path = filepath.replace('/lakehouse/default/', '')
        notebookutils.fs.cp(rel_path, f'file://{local_path}')

    if not Path(local_path).exists():
        raise FileNotFoundError(f"Falta el archivo de configuración en: {filepath}")

    wb = openpyxl.load_workbook(local_path, data_only=True)
    for sheet in wb.worksheets:
        _tmap = {k.lower(): k for k in sheet.tables}
        if table_name.lower() in _tmap:
            table = sheet.tables[_tmap[table_name.lower()]]
            data = sheet[table.ref]
            rows = [[cell.value for cell in row] for row in data]
            if not rows:
                return pd.DataFrame()
            return pd.DataFrame(rows[1:], columns=rows[0])

    raise ValueError(f"No se encontró la tabla '{table_name}' en el archivo {filepath}")


def get_active_plants_dict():
    """Lee el Excel y devuelve un diccionario {plant_code: prefijo_en_mayusculas}"""
    plants_df = read_excel_table(CONFIG_XLSX, 'T01_plants')

    required_cols = ['plant_code', 'pvsyst_file', 'pr_execution']
    for col in required_cols:
        if col not in plants_df.columns:
            raise KeyError(f"Falta la columna '{col}' en T01_plants.")

    # Filtro flexible: acepta "true", "verdadero", "1", "activo", "yes" incluso con símbolos extra
    valores_limpios = plants_df['pr_execution'].astype(str).str.strip().str.lower()
    is_active = valores_limpios.str.contains('true|verdadero|1|si|yes|v|t|activo', regex=True, na=False)

    known = plants_df[is_active].dropna(subset=['plant_code', 'pvsyst_file']).copy()

    known['plant_code'] = known['plant_code'].astype(str).str.strip().str.upper()
    known['pvsyst_file'] = known['pvsyst_file'].astype(str).str.strip().str.upper()

    return dict(zip(known['plant_code'], known['pvsyst_file']))


# 1. Traernos el diccionario de las plantas oficiales
active_plants = get_active_plants_dict()

# 2. Filtrar y re-mapear to_ingest
#    Acepta filenames empezando por plant_code (ej. PL2_20260608.csv) o por
#    pvsyst_file (ej. Plaza-ex_20260608.csv). Cualquiera de los dos identifica
#    la planta. Si solo coincide la carpeta y nada del filename, también lo
#    aceptamos (la carpeta es la fuente autoritativa).
final_to_ingest = []
for c in to_ingest:
    folder_name = str(c['plant_code']).strip().upper()
    filename = str(c['filename']).strip().upper()

    matched_real_code = None
    for real_pcode, prefix in active_plants.items():
        # Match por prefijo pvsyst_file (convención histórica: Plaza-ex_...)
        if filename.startswith(prefix + '_') or filename.startswith(prefix + '.'):
            matched_real_code = real_pcode
            break
        # Match por prefijo plant_code (convención del auto_run.bat: PL2_...)
        if filename.startswith(real_pcode + '_') or filename.startswith(real_pcode + '.'):
            matched_real_code = real_pcode
            break

    # Fallback: si el filename no matchea pero la carpeta SÍ es un plant_code conocido,
    # confiamos en la carpeta (es la fuente autoritativa).
    if matched_real_code is None and folder_name in active_plants:
        matched_real_code = folder_name

    if matched_real_code:
        c['plant_code'] = matched_real_code
        final_to_ingest.append(c)
    else:
        print(f"  [DEBUG] Descartado: '{filename}' en carpeta '{folder_name}'. Ni el filename ni la carpeta coinciden con plantas activas en T01_plants.")

to_ingest = final_to_ingest
print(f'\nPlantas listas para ingerir tras el filtro: {sorted(list(set(c["plant_code"] for c in to_ingest)))}')

## 3 · Reading and writing helpers 

In [9]:
def _read_csv_text(path: str) -> str:
    """Lee TODO el contenido del CSV. Si `open()` no puede leer la URL
    directa (típico de paths abfss:// en Fabric), copia primero a /tmp/
    con notebookutils.fs.cp y lee desde disco local.

    NO usamos notebookutils.fs.head como fallback porque tiene un cap
    interno (~100 KB) que trunca silenciosamente ficheros más grandes,
    perdiendo filas sin avisar.
    """
    # 1º intento: open() directo (funciona para rutas locales /lakehouse/...)
    try:
        with open(path, 'r', encoding='utf-8-sig', errors='replace') as f:
            return f.read()
    except Exception:
        pass

    # 2º intento: copiar a /tmp/ via notebookutils y leer del disco local
    import os
    safe_name = Path(path).name.replace(' ', '_')
    tmp_path  = f'/tmp/_csv_ingest_{os.getpid()}_{safe_name}'
    try:
        notebookutils.fs.cp(path, f'file://{tmp_path}')
        with open(tmp_path, 'r', encoding='utf-8-sig', errors='replace') as f:
            return f.read()
    finally:
        try:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
        except Exception:
            pass


def _parse_pvsyst_dates(series: pd.Series) -> pd.Series:
    """Convierte una serie de strings de fecha PVsyst a datetime, tolerando
    tanto año a 2 dígitos (28/04/26) como a 4 dígitos (28/04/2026)."""
    # Intentamos 2 dígitos primero (formato histórico)
    out = pd.to_datetime(series, format='%d/%m/%y %H:%M', errors='coerce')
    # Las NaT (cosas que no encajaron) las reparseamos con 4 dígitos
    mask_nat = out.isna()
    if mask_nat.any():
        retry = pd.to_datetime(series[mask_nat], format='%d/%m/%Y %H:%M', errors='coerce')
        out.loc[mask_nat] = retry
    # PVsyst >=8 (p.ej. FRA) exporta ISO 'YYYY-MM-DD HH:MM:SS'. Se parsea con el
    # parser por defecto (dayfirst=False) ANTES del fallback dayfirst, porque
    # dayfirst=True sobre una fecha ISO intercambia mes y dia (2026-06-01 -> 2026-01-06).
    mask_nat = out.isna()
    if mask_nat.any():
        retry = pd.to_datetime(series[mask_nat], errors='coerce')
        out.loc[mask_nat] = retry
    # Fallback final: parser automático con dayfirst=True por si hay otros formatos europeos raros
    mask_nat = out.isna()
    if mask_nat.any():
        retry = pd.to_datetime(series[mask_nat], errors='coerce', dayfirst=True)
        out.loc[mask_nat] = retry
    return out


def read_pvsyst_csv(path: str) -> pd.DataFrame:
    """
    Detectamos la línea cuya primera columna == 'date', saltamos la fila
    siguiente (unidades) y leemos lo demás como CSV separado por ';'.
    """
    raw = _read_csv_text(path)
    lines = raw.splitlines()
    header_idx = None
    for i, line in enumerate(lines[:40]):
        first = line.split(';')[0].strip().lower()
        if first == 'date':
            header_idx = i
            break
    if header_idx is None:
        raise ValueError(f'No encuentro la cabecera "date" en {path}')
    body = '\n'.join([lines[header_idx]] + lines[header_idx + 2:]).replace(',', '.')
    df = pd.read_csv(StringIO(body), sep=';', decimal='.', low_memory=False)
    # Parseo de fechas tolerante: PVsyst CLI a veces exporta DD/MM/YY (2 dígitos)
    # y a veces DD/MM/YYYY (4 dígitos). _parse_pvsyst_dates prueba ambos.
    df[df.columns[0]] = _parse_pvsyst_dates(df[df.columns[0]])
    n_total = len(df)
    n_validas = df[df.columns[0]].notna().sum()
    if n_validas < n_total:
        print(f'  [WARN] {n_total - n_validas}/{n_total} fechas no parseadas en {Path(path).name}')
    return df




def detect_resolution(ts: pd.Series) -> pd.Timedelta:
    s = pd.to_datetime(ts, errors='coerce').dropna().sort_values()
    diffs = s.diff().dropna()
    return diffs.mode().iloc[0] if not diffs.empty else pd.Timedelta(0)


def write_raw_pvsyst(df_raw: pd.DataFrame, plant: str, source_file: str) -> int:
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

    table = 'raw_pvsyst'
    df = df_raw.copy()
    # Normalizar la primera columna (PVsyst exporta 'date' en minúscula) a 'Date'
    # para mantener consistencia con scada_raw / scada_proc / proc_pvsyst.
    if df.columns[0] != 'Date':
        df = df.rename(columns={df.columns[0]: 'Date'})
    date_col = 'Date'

    df['plant_code'] = plant
    df['source_file'] = source_file
    df['ingested_at'] = datetime.utcnow()

    for c in df.columns:
        if c not in (date_col, 'plant_code', 'source_file', 'ingested_at'):
            df[c] = df[c].astype(str)

    sdf = spark.createDataFrame(df)

    if spark.catalog.tableExists(table):
        target = DeltaTable.forName(spark, table)
        target.alias('t').merge(
            sdf.alias('s'),
            f"t.plant_code = s.plant_code AND t.`{date_col}` = s.`{date_col}` AND t.source_file = s.source_file"
        ).whenNotMatchedInsertAll().execute()
    else:
        sdf.write.format('delta') \
           .option("mergeSchema", "true") \
           .partitionBy('plant_code') \
           .mode('append') \
           .saveAsTable(table)

    return len(df)


def write_proc_pvsyst(df_clean: pd.DataFrame, plant: str, source_file: str):
    df = df_clean.copy()
    df['plant_code']  = plant
    df['source_file'] = source_file
    df['ingested_at'] = datetime.utcnow()
    canonical = {'plant_code', 'Date', 'GHI', 'DHI', 'T_AMB', 'ALB', 'POA', 'E_Grid',
                 'time_shift', 'source_file', 'ingested_at'}
    drop = [c for c in df.columns if c not in canonical]
    if drop:
        df = df.drop(columns=drop)
    # DHI (difusa, de DiffHor) incluida; si el CSV no la trae queda NaN para mantener
    # consistencia de columnas entre plantas.
    for c in ('GHI', 'DHI', 'T_AMB', 'ALB', 'POA', 'E_Grid'):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        else:
            df[c] = float('nan')
    sdf = spark.createDataFrame(df)
    # autoMerge: añade la columna DHI a proc_pvsyst si aún no existe
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    target = DeltaTable.forName(spark, 'proc_pvsyst')
    target.alias('t').merge(
        sdf.alias('s'),
        't.plant_code = s.plant_code AND t.Date = s.Date'
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


def _pvsyst_shift_rules(plant_code):
    """08_Time_Shifts (T08: plant_code, time_shift, start_date, end_date) -> reglas
    [(start, end, minutes)] para la planta. minutes en MINUTOS, redondeado al
    multiplo de 15 mas cercano (rejilla horaria). Periodo sobre la Date PVSyst."""
    rules = []
    try:
        df = read_excel_table(CONFIG_XLSX, 'T08_time_shifts')
        df.columns = [str(c).strip().lower() for c in df.columns]
        sub = df[df['plant_code'].astype(str).str.strip().str.upper()
                 == str(plant_code).strip().upper()]
        for _, r in sub.iterrows():
            try:
                raw = float(r.get('time_shift'))
            except (TypeError, ValueError):
                continue
            mins = int(round(raw / 15.0)) * 15
            if mins == 0:
                continue
            start = pd.to_datetime(r.get('start_date'), errors='coerce')
            end   = pd.to_datetime(r.get('end_date'),   errors='coerce')
            if pd.isna(start) or pd.isna(end):
                continue
            rules.append((start, end, mins))
    except Exception as e:
        print(f'  [time_shift] no se pudieron leer reglas T08 para {plant_code}: {e}')
    return rules


def apply_pvsyst_time_shifts(df, plant_code, manual_min=0):
    """Desplaza las fechas PVSyst segun T08_time_shifts (por periodo; suma minutos
    a las filas cuya Date cae en [start, end], fin de dia inclusivo). Anade la
    columna 'time_shift' con los minutos aplicados por fila. Si la planta no tiene
    reglas en T08 y manual_min != 0, aplica ese override global (legacy)."""
    if 'Date' not in df.columns or df.empty:
        df = df.copy()
        df['time_shift'] = 0
        return df
    rules = _pvsyst_shift_rules(plant_code)
    orig = pd.to_datetime(df['Date'])
    shift_min = pd.Series(0, index=df.index, dtype='int64')
    if rules:
        for start, end, mins in rules:
            end_excl = pd.Timestamp(end).normalize() + pd.Timedelta(days=1)
            mask = (orig >= pd.Timestamp(start)) & (orig < end_excl)
            shift_min.loc[mask] = mins
    elif manual_min:
        shift_min.loc[:] = int(round(float(manual_min) / 15.0)) * 15
    df = df.copy()
    df['Date'] = orig + pd.to_timedelta(shift_min, unit='m')
    df['time_shift'] = shift_min.values
    df = df.sort_values('Date').reset_index(drop=True)
    applied = sorted({int(x) for x in shift_min.unique() if x != 0})
    if applied:
        n = int((shift_min != 0).sum())
        src = 'T08' if rules else 'override manual'
        print(f'  time_shift ({src}): {applied} min aplicados a {n:,}/{len(df):,} filas PVSyst')
    return df


#### Parsing helper

In [10]:
_PVSYST_RENAME = {
    'date':    'Date',
    'globhor': 'GHI',
    'diffhor': 'DHI',
    't_amb':   'T_AMB',
    'tamb':    'T_AMB',
    'windvel': 'WS',
    'globinc': 'POA',
    'globeff': 'POA',
    'alb_gl':  'ALB',
    'albedo':  'ALB',
    'tarray':  'T_ARRAY',
    'e_grid':  'E_Grid',
    'eoutinv': 'E_Grid',
}


def normalize_pvsyst(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    # Si el CSV contiene GlobInc y GlobEff, conservar solo GlobEff (irradiancia efectiva
    # tras pérdidas ópticas) para evitar columna POA duplicada al renombrar.
    cols_lower = {c.lower().strip(): c for c in df.columns}
    if 'globinc' in cols_lower and 'globeff' in cols_lower:
        df = df.drop(columns=[cols_lower['globinc']])
    rename = {}
    for c in df.columns:
        cl = str(c).lower().strip()
        cl_short = cl.split(' ')[0]
        target = _PVSYST_RENAME.get(cl, _PVSYST_RENAME.get(cl_short))
        if target:
            rename[c] = target
    df = df.rename(columns=rename)
    if 'Date' not in df.columns:
        df = df.rename(columns={df.columns[0]: 'Date'})
    # Si Date ya viene como datetime de read_pvsyst_csv, pd.to_datetime no rompe.
    # Si viene como string (caso edge), re-parseamos con tolerancia (2 ó 4 dígitos).
    if not pd.api.types.is_datetime64_any_dtype(df['Date']):
        df['Date'] = _parse_pvsyst_dates(df['Date'])
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
    return df

## 4 · Main loop - one file each time

In [11]:
results = []
for c in to_ingest:
    plant = c['plant_code']; path = c['full_path']
    print(f"\n→ {plant}: {c['filename']}")
    status = 'ok'
    error = None
    period_start = period_end = None
    resolution = None
    row_count = 0
    df_raw = None

    # --- Paso 1/3: leer el CSV ---------------------------------------
    try:
        df_raw = read_pvsyst_csv(path)
        print(f'  [1/3] CSV leído  · {len(df_raw):,} filas')
    except Exception as e:
        status = 'failed'
        error  = f'read_pvsyst_csv: {str(e)[:400]}'
        print(f'  ✗ FALLO en lectura: {error}')

    # --- Paso 2/3: escribir RAW (SIEMPRE antes que PROC) -------------
    # Si raw falla, no intentamos proc. Si raw OK pero proc falla luego,
    # raw queda guardado igualmente y marcamos status='partial'.
    if df_raw is not None and status == 'ok':
        try:
            row_count = write_raw_pvsyst(df_raw, plant, c['filename'])
            print(f'  [2/3] raw_pvsyst  ← {row_count:,} filas')
        except Exception as e:
            status = 'failed'
            error  = f'write_raw_pvsyst: {str(e)[:400]}'
            print(f'  ✗ FALLO escribiendo raw: {error}')

    # --- Paso 3/3: normalizar + escribir PROC ------------------------
    # Si esto falla, raw ya está persistido. Marcamos 'partial' para que
    # el próximo csv_ingest lo reintente (el filtro es status='ok').
    if df_raw is not None and status == 'ok':
        try:
            df_clean = normalize_pvsyst(df_raw)
            # time-shift por planta/periodo desde T08_time_shifts (-> columna time_shift)
            df_clean = apply_pvsyst_time_shifts(df_clean, plant, manual_min=time_shift_min)
            write_proc_pvsyst(df_clean, plant, c['filename'])
            period_start = df_clean['Date'].min()
            period_end   = df_clean['Date'].max()
            resolution   = str(detect_resolution(df_clean['Date']))
            print(f'  [3/3] proc_pvsyst ← {len(df_clean):,} filas · resolución {resolution}')
        except Exception as e:
            status = 'partial'
            error  = f'write_proc_pvsyst: {str(e)[:400]}'
            print(f'  ⚠ PARCIAL: raw OK pero proc FALLÓ: {error}')

    results.append(Row(
        plant_code=plant, file_path=path, file_kind='pvsyst',
        file_size=c['file_size'], row_count=row_count,
        period_start=period_start, period_end=period_end,
        resolution=resolution, status=status, error=error,
        ingested_at=datetime.utcnow(),
    ))

if results:
    # Convertimos a pandas primero para que None → NaT en columnas timestamp
    # y Spark pueda inferir los tipos sin CANNOT_DETERMINE_TYPE.
    rows_pd = pd.DataFrame([r.asDict() for r in results])
    for col in ('period_start', 'period_end', 'ingested_at'):
        rows_pd[col] = pd.to_datetime(rows_pd[col], errors='coerce')
    spark.createDataFrame(rows_pd).write.mode('append').format('delta').saveAsTable('ingested_files')

    n_ok      = sum(1 for r in results if r['status'] == 'ok')
    n_partial = sum(1 for r in results if r['status'] == 'partial')
    n_failed  = sum(1 for r in results if r['status'] == 'failed')
    print(f'\nLogged {len(results)} filas en ingested_files  ·  ok={n_ok}  partial={n_partial}  failed={n_failed}')


In [13]:
summary = {
    'candidates_found': len(candidates),
    'newly_ingested':   sum(1 for r in results if r['status'] == 'ok'),
    'failed':           sum(1 for r in results if r['status'] == 'failed'),
}
print(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary))
